# Chest X-ray diagnosis and disease progression

This notebook labels current studies, prepares consecutive image pairs, and compares three progression experiments on the same patient splits.

| Experiment | Initialization | Training |
|---|---|---|
| Frozen-feature baseline | Saved diagnosis checkpoint | Frozen DenseNet → feature difference → PCA/logistic regression per disease |
| Paired fine-tuning | Saved diagnosis checkpoint | Train DenseNet and a five-disease progression head together |
| Paired training from scratch | Random weights; no ImageNet or diagnosis weights | Train the same paired architecture and head together |

Each pair uses one frontal-preferred image from each study. Targets are improving, stable, and worsening. Mixed/no-comparison targets are ignored per disease. Edema and Pulmonary edema overlap in the current labeling rules. These are automatic candidate labels requiring validation.

Run setup and data preparation first. Enable the experiment switches you want in Section 1. Long training runs are disabled by default. Saved outputs from the original notebook are not copied into this clean notebook.


## 1. Imports, paths, and experiment settings

Install missing dependencies in a separate cell if needed: `%pip install scikit-learn joblib tqdm pandas numpy pydicom pillow matplotlib`. Keep the existing CUDA-enabled PyTorch/torchvision installation.


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd
import torch
from IPython.display import display

PROJECT_DIR = Path(r"C:\Users\Zila\Downloads\BN5212\midterm_prognosis")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


In [2]:
from check_number_of_files import patients_with_more_than_1_txt
from labelling.Build_disease_vocab import (
    load_icd_diagnoses, load_icd_to_phecode_mapping, map_icd_to_phecode, drop_unmapped,
)
from labelling.scan_disease_v2 import main as label_reports
from labelling.count_positive_cases import main as count_positive_cases
from labelling.build_prognosis_labels import main as build_progression_labels
from data_loader.cxr_backbone_dataloader import make_backbone_dataloaders
from data_loader.cxr_pair_dataloader import prepare_pair_csv, make_pair_loaders
from model.cxr_backbone import CXRBackbone
from model.train_backbone import train_backbone
from model.train_progression import train_progression
from model.train_paired_progression import train_paired_progression


In [ ]:
REPORTS_ROOT = PROJECT_DIR.parent / "MIMIC-CXR" / "p10" / "p10"
LABELLING_DIR = PROJECT_DIR / "labelling"
ICD_PATH = PROJECT_DIR.parent / "MIMIC-IV" / "3.1" / "hosp" / "d_icd_diagnoses.csv"
VOCAB_PATH = LABELLING_DIR / "disease_vocab" / "unique_diseases.csv"
PATIENTS_PATH = LABELLING_DIR / "cxr_labels" / "patients_with_more_than_1_txt.csv"
DIAGNOSIS_DIR = LABELLING_DIR / "cxr_labels_selected_everpositive"
LABELS_PATH = DIAGNOSIS_DIR / "study_labels.csv"
REVIEWS_PATH = DIAGNOSIS_DIR / "review_queue.csv"
COUNTS_PATH = DIAGNOSIS_DIR / "disease_positive_counts.csv"
HYBRID_DIR = LABELLING_DIR / "prognosis_labels_hybrid"
PAIR_CSV = HYBRID_DIR / "all_disease_pairs.csv"
SPLIT_CSV = LABELLING_DIR / "prognosis_patient_splits.csv"
CHECKPOINT_PATH = PROJECT_DIR / "training_outputs" / "backbone_20261003_150057" / "best_backbone.pt"

DISEASES = ["Pleural effusion", "Cardiomegaly", "Edema", "Pneumonia", "Pulmonary edema"]
CLASS_NAMES = ["improving", "stable", "worsening"]
SEED = 42
EPOCHS = 20
PAIR_BATCH_SIZE = 8
LEARNING_RATE = 1e-4  # Diagnosis-backbone training
FINETUNE_BACKBONE_LR = 1e-5
FINETUNE_HEAD_LR = 1e-4
SCRATCH_BACKBONE_LR = 1e-4
SCRATCH_HEAD_LR = 1e-4
FINAL_LR_RATIO = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Reuse existing labels and splits; set rebuild=True only when intentionally updating them.
REBUILD_LABELS = False
REBUILD_PAIRS = False
USE_CANDIDATES = True
RUN_BACKBONE = False
RUN_FROZEN_BASELINE = False
RUN_PAIRED_FINETUNE = True #I turned this true
RUN_PAIRED_SCRATCH = False

def new_run(name):
    return PROJECT_DIR / "training_outputs" / datetime.now().strftime(name + "_%Y%m%d_%H%M%S_%f")

print("Device:", DEVICE)
print("Diagnosis checkpoint:", CHECKPOINT_PATH)


## 2. Cohort and disease vocabulary

The cohort helper selects patients with at least two report files. Later pairing uses study timestamps and excludes studies that cannot be ordered reliably. Multiple DICOM files from one examination do not count as separate time points.


In [4]:
if not PATIENTS_PATH.exists():
    count, patient_ids = patients_with_more_than_1_txt(REPORTS_ROOT)
    PATIENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"Patient": patient_ids}).to_csv(PATIENTS_PATH, index=False)
patients_table = pd.read_csv(PATIENTS_PATH)
print("Selected patients:", len(patients_table))

if not VOCAB_PATH.exists():
    diagnoses = load_icd_diagnoses(ICD_PATH)
    mapping, phenotype_names = load_icd_to_phecode_mapping()
    mapped = map_icd_to_phecode(diagnoses, mapping, phenotype_names)
    disease_names = drop_unmapped(mapped)
    VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"disease": disease_names}).to_csv(VOCAB_PATH, index=False)
print("Vocabulary entries:", len(pd.read_csv(VOCAB_PATH)))


Selected patients: 506
Vocabulary entries: 3544


## 3. Disease-presence and progression labels

Scan Findings/Impression for disease presence. Build progression candidates using comparison phrases and explicit positive/negative transitions. The paired task includes both positive-input cases and negative-to-positive onset cases. Missing mentions do not imply resolution.

Rebuilding labels can replace generated review queues; preserve manual review files first. Existing files are reused by default.


In [5]:
if REBUILD_LABELS or not LABELS_PATH.exists():
    label_reports([
        "--vocab", str(VOCAB_PATH), "--patients", str(PATIENTS_PATH),
        "--reports", str(REPORTS_ROOT), "--skip-uncertain-after-positive", "--out", str(DIAGNOSIS_DIR),
    ])
if REBUILD_LABELS or not COUNTS_PATH.exists():
    count_positive_cases([
        "--labels", str(LABELS_PATH), "--reviews", str(REVIEWS_PATH),
        "--vocab", str(VOCAB_PATH), "--out", str(COUNTS_PATH),
    ])
diagnosis_counts = pd.read_csv(COUNTS_PATH)
display(diagnosis_counts[diagnosis_counts["disease"].isin(DISEASES)])

if REBUILD_LABELS or not (HYBRID_DIR / "prognosis_pairs.csv").exists():
    build_progression_labels([
        "--reports", str(REPORTS_ROOT), "--patients", str(PATIENTS_PATH),
        "--labels", str(LABELS_PATH), "--reviews", str(REVIEWS_PATH),
        "--mode", "lenient", "--out", str(HYBRID_DIR),
    ])
progression_pairs = pd.read_csv(HYBRID_DIR / "prognosis_pairs.csv")
display(pd.crosstab(progression_pairs["disease"], progression_pairs["candidate_target"]).reindex(
    index=DISEASES, columns=CLASS_NAMES + ["mixed", "no_comparison"], fill_value=0,
))


,disease,positive_patient_count,positive_report_count
0,Pleural effusion,167,415
1,Cardiomegaly,155,364
2,Edema,139,363
3,Pneumonia,130,225
4,Pulmonary edema,119,277


candidate_target,improving,stable,worsening,mixed,no_comparison
disease,,,,,
Pleural effusion,94,140,180,27,1910
Cardiomegaly,17,406,39,2,1887
Edema,100,40,87,8,2116
Pneumonia,41,11,77,1,2221
Pulmonary edema,93,41,84,7,2126


### Disease-presence distribution

Each X-ray visit means one report/study, not the number of DICOM files. Patients are counted once per disease. All diseases with positive visits are shown; the five selected diseases have distinct colors.


In [ ]:
# Include every disease with at least one positive visit.
# Five selected diseases have distinct colors; other diseases are gray.
disease_colors = dict(zip(DISEASES, ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00"]))
positive_counts = diagnosis_counts.loc[diagnosis_counts["positive_report_count"] > 0].copy()
positive_counts = positive_counts.sort_values("positive_report_count", ascending=False)
if positive_counts.empty:
    print("No positive disease counts to plot.")
else:
    colors = [disease_colors.get(d, "#BDBDBD") for d in positive_counts["disease"]]
    fig, axes = plt.subplots(1, 2, figsize=(15, max(5, len(positive_counts) * 0.30)), sharey=True, layout="constrained")
    for ax, column, title in zip(axes, ["positive_patient_count", "positive_report_count"],
                                ["Patients with a positive label", "X-ray visits with a positive label"]):
        bars = ax.barh(positive_counts["disease"], positive_counts[column], color=colors)
        ax.bar_label(bars, padding=3, fontsize=8)
        ax.set_title(title)
        ax.set_xlabel("Count")
        ax.set_xlim(0, max(positive_counts[column].max() * 1.18, 1))
        ax.grid(axis="x", alpha=0.2)
        ax.set_axisbelow(True)
    axes[0].invert_yaxis()
    axes[1].legend(handles=[Patch(color=disease_colors[d], label=d) for d in DISEASES]
                   + [Patch(color="#BDBDBD", label="Other diseases")], loc="lower right")
    fig.suptitle("Disease presence: patients and X-ray visits", fontsize=15)
    plt.show()


## 4. Paired train, validation, and test loaders

Reuse the backbone's patient split CSV. Every unique pair has five labels; `-100` masks a disease without a usable label. Each model sees the same pair manifest. The DICOM loader currently decodes images on demand without an image cache.


In [6]:
if not SPLIT_CSV.is_file():
    raise FileNotFoundError(f"Existing backbone patient splits are required: {SPLIT_CSV}")
if REBUILD_PAIRS or REBUILD_LABELS or not PAIR_CSV.exists():
    prepare_pair_csv(
        pairs_csv=HYBRID_DIR / "prognosis_pairs.csv",
        split_csv=SPLIT_CSV, reports_root=REPORTS_ROOT, output_csv=PAIR_CSV,
        use_candidates=USE_CANDIDATES,
    )
pair_loaders = make_pair_loaders(PAIR_CSV, batch_size=PAIR_BATCH_SIZE)
pair_manifest = pd.read_csv(PAIR_CSV)
pair_summary = []
for split, group in pair_manifest.groupby("split"):
    for disease in DISEASES:
        counts = group[disease].value_counts()
        pair_summary.append({"split": split, "disease": disease,
                             **{label: int(counts.get(i, 0)) for i, label in enumerate(CLASS_NAMES)},
                             "masked": int(counts.get(-100, 0))})
display(pd.DataFrame(pair_summary))


train: 704 pairs, 212 patients
val: 92 pairs, 40 patients
test: 119 pairs, 39 patients


,split,disease,improving,stable,worsening,masked
0,test,Pleural effusion,10,24,21,64
1,test,Cardiomegaly,1,47,9,62
2,test,Edema,12,6,10,91
3,test,Pneumonia,7,0,5,107
4,test,Pulmonary edema,11,6,10,92
5,train,Pleural effusion,75,111,145,373
6,train,Cardiomegaly,15,311,25,353
7,train,Edema,77,32,66,529
8,train,Pneumonia,29,8,64,603
9,train,Pulmonary edema,71,33,65,535


### Progression-label distribution

Counts come from the actual pair manifest, across all splits. Masked labels (`-100`), including unusable mixed/no-comparison targets, are excluded. A pair can contribute to several diseases.


In [ ]:
# Count the exact labels passed to the loaders, across all patient splits.
progression_plot_counts = pd.DataFrame({
    disease: [(pair_manifest[disease] == i).sum() for i in range(len(CLASS_NAMES))]
    for disease in DISEASES
}, index=CLASS_NAMES).T
if pair_manifest.duplicated(["subject_id", "input_study_id", "target_study_id"]).any():
    raise ValueError("Duplicate study pairs in the manifest; check before counting.")
display(progression_plot_counts)
ax = progression_plot_counts.plot.bar(
    figsize=(11, 5), color=["#009E73", "#0072B2", "#D55E00"], rot=15,
)
for bars in ax.containers:
    ax.bar_label(bars, padding=3, fontsize=9)
ax.set(title="Disease-specific progression labels: all train/validation/test pairs",
       xlabel="Disease", ylabel="Number of labeled X-ray pairs")
ax.margins(y=0.18)
ax.legend(title="Progression")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


## 5. Optional diagnosis-backbone training

Enable `RUN_BACKBONE` to train the original five-disease, four-state diagnosis model. Otherwise use the existing checkpoint. This is separate from paired progression training.


In [7]:
if RUN_BACKBONE:
    backbone_loaders = make_backbone_dataloaders(
        labels_csv=LABELS_PATH, reviews_csv=REVIEWS_PATH,
        report_summary_csv=DIAGNOSIS_DIR / "report_summary.csv", split_csv=SPLIT_CSV,
        reports_root=REPORTS_ROOT, batch_size=16, num_workers=0,
        manifest_path=LABELLING_DIR / "backbone_image_manifest.csv",
    )
    torch.manual_seed(SEED)
    diagnosis_model = CXRBackbone(pretrained=True)
    backbone_results = train_backbone(
        diagnosis_model, backbone_loaders, new_run("backbone"), epochs=EPOCHS,
        learning_rate=LEARNING_RATE, seed=SEED, device=DEVICE,
    )
    CHECKPOINT_PATH = backbone_results["best_path"]
    diagnosis_model.cpu()
    del diagnosis_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("Using existing diagnosis checkpoint:", CHECKPOINT_PATH)


Using existing diagnosis checkpoint: C:\Users\Zila\Downloads\BN5212\midterm_prognosis\training_outputs\backbone_20261003_150057\best_backbone.pt


## 6. Frozen-feature baseline: diagnosis checkpoint + PCA + logistic regression

This reproduces Roger's feature-difference approach with our five disease-specific three-class targets. DenseNet stays frozen; each disease gets its own PCA and classifier. Validation selects regularization. This is a baseline, not end-to-end fine-tuning.


In [8]:
if RUN_FROZEN_BASELINE:
    frozen_results = train_progression(
        checkpoint_path=CHECKPOINT_PATH, pair_loaders=pair_loaders,
        out_dir=new_run("paired_frozen_pca"), pca_components=50,
        c_values=(0.1, 0.5, 1.0, 10.0), seed=SEED, device=DEVICE,
    )
    display(pd.read_csv(frozen_results["out_dir"] / "metrics.csv"))
else:
    print("Set RUN_FROZEN_BASELINE=True to run this experiment.")


Set RUN_FROZEN_BASELINE=True to run this experiment.


## 7. Paired progression fine-tuning from the diagnosis checkpoint

Load the diagnosis encoder weights, attach a fresh five-disease progression head, and update **all encoder and head weights** on image pairs. Both images pass through the shared DenseNet; their feature difference produces a `[batch, 5, 3]` prediction tensor. Use masked cross-entropy and select the best checkpoint by validation loss.


Learning rates start at `1e-5` for the backbone and `1e-4` for the head. Cosine decay reaches 10% of each rate in the final epoch. Each epoch records both rates in `history.csv`.


In [ ]:
if RUN_PAIRED_FINETUNE:
    finetune_results = train_paired_progression(
        pair_loaders=pair_loaders, out_dir=new_run("paired_finetune"),
        diagnosis_checkpoint=CHECKPOINT_PATH, epochs=EPOCHS,
        backbone_lr=FINETUNE_BACKBONE_LR, head_lr=FINETUNE_HEAD_LR,
        final_lr_ratio=FINAL_LR_RATIO, seed=SEED, device=DEVICE,
    )
    display(pd.DataFrame(finetune_results["metrics"]))
else:
    print("Set RUN_PAIRED_FINETUNE=True to fine-tune from the diagnosis checkpoint.")


## 8. Paired progression training from scratch

Use the **same trainable architecture, labels, patient splits, and epoch count** as Section 7, starting with random weights. No diagnosis checkpoint or ImageNet weights are loaded. Both learning rates start at `1e-4` and decay to `1e-5`. This comparison differs in both initialization and backbone learning rate; it is not an initialization-only ablation. Each experiment resets its seed and training-loader shuffle generator.


In [ ]:
if RUN_PAIRED_SCRATCH:
    scratch_results = train_paired_progression(
        pair_loaders=pair_loaders, out_dir=new_run("paired_scratch"),
        diagnosis_checkpoint=None, epochs=EPOCHS,
        backbone_lr=SCRATCH_BACKBONE_LR, head_lr=SCRATCH_HEAD_LR,
        final_lr_ratio=FINAL_LR_RATIO, seed=SEED, device=DEVICE,
    )
    display(pd.DataFrame(scratch_results["metrics"]))
else:
    print("Set RUN_PAIRED_SCRATCH=True to train the paired model from random weights.")


## 9. Compare completed experiments

Use validation results to make model choices. Test results are held-out descriptions, not a tuning signal. Macro F1 includes all three classes; consult class supports where a disease has few or no examples of one class. This table compares frozen classifiers with two end-to-end neural models, so the frozen baseline also differs in architecture/training method.


In [ ]:
comparison_tables = []
for result_name, experiment in [
    ("frozen_results", "Frozen diagnosis features + PCA/LR"),
    ("finetune_results", "Paired fine-tuning from diagnosis"),
    ("scratch_results", "Paired training from scratch"),
]:
    result = globals().get(result_name)
    if result is not None:
        table = pd.read_csv(result["out_dir"] / "metrics.csv")
        table.insert(0, "experiment", experiment)
        comparison_tables.append(table)
if comparison_tables:
    experiment_comparison = pd.concat(comparison_tables, ignore_index=True)
    display(experiment_comparison[["experiment", "disease", "split", "n_pairs", "accuracy", "macro_f1"]])
else:
    print("Enable and run an experiment to populate this table.")


## 10. Experiment confusion matrices

Load saved validation/test evaluations without training again. Each panel is disease-specific: rows are actual labels, columns are predictions. Cells show counts and percentages within the actual class; N/A means that class has no examples. Colors use the same 0–100% scale in every panel. The selected run folder is printed; override it below to compare particular runs.


In [ ]:
# Optional: set a folder explicitly to compare specific saved runs.
# None uses this session's result, then the latest completed saved run.
CONFUSION_RUN_DIRS = {
    "Frozen diagnosis features + PCA/LR": None,
    "Paired fine-tuning from diagnosis": None,
    "Paired training from scratch": None,
}
CONFUSION_SPLITS = ["val", "test"]
experiment_sources = [
    ("Frozen diagnosis features + PCA/LR", "frozen_results", "paired_frozen_pca_*", "metrics.json"),
    ("Paired fine-tuning from diagnosis", "finetune_results", "paired_finetune_*", "confusion_matrices.json"),
    ("Paired training from scratch", "scratch_results", "paired_scratch_*", "confusion_matrices.json"),
]
for experiment, result_name, pattern, filename in experiment_sources:
    run_dir = CONFUSION_RUN_DIRS[experiment]
    if run_dir is None:
        result = globals().get(result_name)
        if result is not None:
            run_dir = result["out_dir"]
        else:
            completed = sorted(p for p in (PROJECT_DIR / "training_outputs").glob(pattern)
                               if (p / filename).is_file())
            run_dir = completed[-1] if completed else None
    if run_dir is None:
        print(f"{experiment}: no completed results yet.")
        continue
    run_dir = Path(run_dir)
    source = run_dir / filename
    if not source.is_file():
        print(f"Missing saved results: {source}")
        continue
    print(f"{experiment}: {run_dir}")
    saved = json.loads(source.read_text(encoding="utf-8"))
    settings_file = run_dir / "settings.json"
    settings = json.loads(settings_file.read_text(encoding="utf-8")) if settings_file.exists() else {}
    saved_diseases = settings.get("diseases", DISEASES)
    saved_classes = settings.get("labels", CLASS_NAMES)
    if list(saved_classes) != CLASS_NAMES:
        raise ValueError(f"Unexpected class order in {run_dir}: {saved_classes}")
    for split in CONFUSION_SPLITS:
        fig, axes = plt.subplots(1, len(DISEASES), figsize=(20, 4), layout="constrained", squeeze=False)
        for ax, disease in zip(axes[0], DISEASES):
            if filename == "metrics.json":
                matrix = saved.get("diseases", {}).get(disease, {}).get(split, {}).get("confusion_matrix")
            else:
                matrices = saved.get(split, {}).get("confusion")
                matrix = matrices[saved_diseases.index(disease)] if matrices is not None and disease in saved_diseases else None
            if matrix is None:
                ax.set_title(disease)
                ax.text(0.5, 0.5, "No saved evaluation", ha="center", va="center")
                ax.set_axis_off()
                continue
            matrix = np.asarray(matrix)
            if matrix.shape != (3, 3):
                raise ValueError(f"Invalid confusion matrix for {experiment}, {disease}, {split}")
            support = matrix.sum(axis=1, keepdims=True)
            fractions = np.divide(matrix, support, out=np.zeros((3, 3), dtype=float), where=support != 0)
            ax.imshow(fractions, vmin=0, vmax=1, cmap="Blues")
            for i in range(3):
                for j in range(3):
                    annotation = f"{int(matrix[i, j])}\n{fractions[i, j]:.0%}" if support[i, 0] else "0\nN/A"
                    ax.text(j, i, annotation, ha="center", va="center",
                            color="white" if fractions[i, j] > 0.5 else "black", fontsize=9)
            ax.set_xticks(range(3), CLASS_NAMES, rotation=45, ha="right")
            ax.set_yticks(range(3), CLASS_NAMES)
            ax.set(title=f"{disease}\nn = {int(matrix.sum())}", xlabel="Predicted label", ylabel="Actual label")
        fig.suptitle(f"{experiment} | {split} | counts and row percentages", fontsize=14)
        plt.show()
